In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.sparse import csr_matrix, lil_matrix

Global variables

In [2]:
DATASET_FOLDER = "../datasets/"
MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 20
TRAIN_RATIO = 0.6
VAL_RATIO = 0.2
TEST_RATIO = 0.2
FOLD_IN_RATIO = 0.8

Load the dataset and replace the user and item ids so they are a contiguous sequence.

In [ ]:
def LoadInteractions():
    train_interactions = pd.read_csv(f"{DATASET_FOLDER}/train_interactions.csv", dtype={"user_id": "int64", "item_id": "int64"})

    train_interactions.rename(columns={'user_id': 'old_user_id', 'item_id': 'old_item_id'}, inplace=True)

    user_id_mapping = {val: i for i, val in enumerate(train_interactions['old_user_id'].unique())}
    train_interactions['user_id'] = train_interactions['old_user_id'].map(user_id_mapping)

    item_id_mapping = {val: i for i, val in enumerate(train_interactions['old_item_id'].unique())}
    train_interactions['item_id'] = train_interactions['old_item_id'].map(item_id_mapping)

    new_to_old_user_id_mapping = {v: k for k, v in user_id_mapping.items()}
    new_to_old_item_id_mapping = {v: k for k, v in item_id_mapping.items()}

    return train_interactions, new_to_old_user_id_mapping, new_to_old_item_id_mapping

def LoadTestData(item_id_mapping):
    test_interactions = pd.read_csv(f"{DATASET_FOLDER}/test_interactions_in.csv")

    test_interactions.rename(columns={'user_id': 'old_user_id', 'item_id': 'old_item_id'}, inplace=True)

    test_interactions["item_id"] = (test_interactions["old_item_id"].map(item_id_mapping))

    test_interactions = test_interactions.dropna(subset=["item_id"]).copy()
    test_interactions["item_id"] = test_interactions["item_id"].astype(int)

    user_id_mapping = {val: i for i, val in enumerate(test_interactions['old_user_id'].unique())}
    test_interactions['user_id'] = test_interactions['old_user_id'].map(user_id_mapping)
    user_id_mapping = {v: k for k, v in user_id_mapping.items()}

    return test_interactions, user_id_mapping

Converts the interactions to a sparse interaction matrix

In [4]:
def CreateCSRMatrix(interactions, num_items=None):

    num_users = interactions['user_id'].unique().size
    if num_items is None:
        num_items = interactions['item_id'].unique().size
    rows = interactions['user_id']
    cols = interactions['item_id']
    data = [1]*interactions['user_id'].size

    interaction_matrix_csr = csr_matrix((data, (rows, cols)), shape=(num_users, num_items))

    return interaction_matrix_csr

Two filter function to filter out users with low interactions and items with low interactions

In [ ]:
def MinUsersPerItem(csr_matrix, item_mapping, min_users):
    # Filter out items with strictly less than min_users interactions
    item_mask = csr_matrix.getnnz(axis=0) >= min_users
    # Hint: use the .getnnz() method of the scipy csr matrix.

    # Apply the mask to the matrix to filter items
    filtered_interaction_matrix = csr_matrix[:, item_mask]

    # Great, because we're removing items (columns) from the matrix, we break our original mapping between columns and "old" item IDs!
    # Update the mapping:
    kept_cols = np.where(item_mask)[0]
    updated_item_mapping = {
        item_mapping[col.astype(int)]: new_idx
        for new_idx, col in enumerate(kept_cols)
    }
    
    return filtered_interaction_matrix, updated_item_mapping

def MinItemsPerUser(interaction_matrix_csr, user_mapping, min_items):
    # Filter out users with strictly less than min_items interactions
    user_mask = interaction_matrix_csr.getnnz(axis=1) >= min_items
    # Hint: use the .getnnz() method of the scipy csr matrix.
    
    # Apply the mask to the matrix to filter users
    filtered_interaction_matrix = interaction_matrix_csr[user_mask]

    # Great, because we're removing users (rows) from the matrix, we break our original mapping between rows and "old" user IDs!
    # Update the mapping:
    kept_rows = np.where(user_mask)[0]
    updated_user_mapping = {
        user_mapping[row]: new_idx
        for new_idx, row in enumerate(kept_rows)
    }
    
    return filtered_interaction_matrix, updated_user_mapping

def ApplyFilters(csr_matrix, user_mapping, item_mapping):
    filtered_matrix, updated_user_mapping = MinItemsPerUser(csr_matrix, user_mapping, MIN_USER_INTERACTIONS)
    filtered_matrix, updated_item_mapping = MinUsersPerItem(filtered_matrix, item_mapping, MIN_ITEM_INTERACTIONS)

    return filtered_matrix, updated_user_mapping, updated_item_mapping

In [6]:
class StrongGeneralizationSplitter:
    def __init__(self, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO, test_ratio=TEST_RATIO, fold_in_ratio=FOLD_IN_RATIO):
        assert train_ratio + val_ratio + test_ratio == 1, "Train, validation, and test ratios must sum to 1."
        self.train_ratio = train_ratio
        self.val_ratio = val_ratio
        self.test_ratio = test_ratio
        self.fold_in_ratio = fold_in_ratio
        self.hold_out_ratio = 1 - fold_in_ratio

    def split(self, interaction_matrix_csr):
        num_users, num_items = interaction_matrix_csr.shape
        
        # Shuffle users for random splitting
        users = np.arange(num_users)
        np.random.shuffle(users)
        
        # Split users into training, validation, and test sets based on the specified ratios
        train_end = int(len(users)*self.train_ratio)
        val_end = train_end + int(len(users)*self.val_ratio)
        # Hint: this is where you'd use your train and validation ratios.

        train_users = users[:train_end]
        val_users = users[train_end:val_end]
        test_users = users[val_end:]
        # Hint: this is where you should slice the "users"
        
        train_matrix = interaction_matrix_csr[train_users, :]
        val_matrix = interaction_matrix_csr[val_users, :]
        test_matrix = interaction_matrix_csr[test_users, :]
        
        # Create fold-in and hold-out splits for validation and test sets
        val_fold_in, val_hold_out = self.split_interactions(val_matrix)
        test_fold_in, test_hold_out  = self.split_interactions(test_matrix)
        
        return train_matrix, (val_fold_in, val_hold_out), (test_fold_in, test_hold_out), (train_users, val_users, test_users)

    def split_interactions(self, interaction_matrix_csr):
        # Convert the matrix to lil format for easier manipulation
        lil = interaction_matrix_csr.tolil()
    
        # Initialize the lists to store fold-in and hold-out data
        fold_in_lil = lil_matrix((interaction_matrix_csr.shape[0], interaction_matrix_csr.shape[1]))
        hold_out_lil = lil_matrix((interaction_matrix_csr.shape[0], interaction_matrix_csr.shape[1]))
    
        # Iterate over each row (user)
        for i in tqdm(range(lil.shape[0]), desc="Splitting interactions: "):
            
            # Get the non-zero entries in this row
            row_data = lil.data[i]
            row_indices = lil.rows[i]
            
            # Determine the number of interactions to include in fold-in
            fold_in_size = int(len(row_data) * self.fold_in_ratio)
            
            # Randomly shuffle the indices
            shuffle_indices = np.random.permutation(len(row_data))
            
            # Split the indices into fold-in and fold-out
            fold_in_indices = shuffle_indices[:fold_in_size]
            hold_out_indices = shuffle_indices[fold_in_size:]
            
            # Assign the fold-in data to the fold_in_lil matrix
            fold_in_lil.rows[i] = np.array(row_indices)[fold_in_indices].tolist()
            fold_in_lil.data[i] = np.array(row_data)[fold_in_indices].tolist()
            
            # Assign the fold-out data to the hold_out_lil matrix
            hold_out_lil.rows[i] = np.array(row_indices)[hold_out_indices].tolist()
            hold_out_lil.data[i] = np.array(row_data)[hold_out_indices].tolist()
    
        # Convert the lil matrices back to csr format
        fold_in_matrix = fold_in_lil.tocsr()
        hold_out_matrix = hold_out_lil.tocsr()
    
        return fold_in_matrix, hold_out_matrix

Final function that combines all the above to be used in other notebooks

In [ ]:
def GetProcessedData():
    train_interactions, user_mapping, item_mapping = LoadInteractions()
    interaction_matrix_csr = CreateCSRMatrix(train_interactions)
    interaction_matrix_csr, user_mapping, item_mapping = ApplyFilters(interaction_matrix_csr, user_mapping, item_mapping)
    splitter = StrongGeneralizationSplitter()
    train_matrix, (val_fold_in, val_hold_out), (test_fold_in, test_hold_out), (train_users, val_users, test_users) = splitter.split(interaction_matrix_csr)
    return train_matrix, (val_fold_in, val_hold_out), (test_fold_in, test_hold_out), (train_users, val_users, test_users)

def GetCodaBenchTestData():
    train_interactions, user_mapping, item_mapping = LoadInteractions()
    interaction_matrix_csr = CreateCSRMatrix(train_interactions)
    #Should be added but messes up item mapping for some reason
    #interaction_matrix_csr, user_mapping, item_mapping = ApplyFilters(interaction_matrix_csr, user_mapping, item_mapping)
    item_mapping = {v: k for k, v in item_mapping.items()}
    test_interactions, user_mapping_test = LoadTestData(item_mapping)
    train_matrix =  CreateCSRMatrix(train_interactions)
    test_matrix = CreateCSRMatrix(test_interactions, num_items=train_matrix.shape[1])
    item_mapping = {v: k for k, v in item_mapping.items()}

    return test_matrix, train_matrix, item_mapping, user_mapping_test, user_mapping
